# E9 Multistage Training

Author: Arush Arora

## Introduction
This codebase has mostly consisted of additive Graph Positional Encodings (GREPs) injections to provide nodal embeddings to the LLMs at hand. This new multi-stage training will rely on training an R-PEARL/GT to simply replicate the graph before expecting it to serve the LLM with **multiplicative** GREPs, which will be factored directly into the attention-mask matrix for rendition to the LLM (as a Hadamard product cover on the attention logits). Thus, the system will be more carefully trained to incorporate the variation in model architecture among GNNs and LLMs (in terms of their pre-trained weights rather than simply their mathematical foundations).

## Mathematical Overview

### The R-PEARL GNN

The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $\renewcommand{\utilde}[1]{\underset{\sim}{#1}}A$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

#### Graph Convolutional Network (GNN)
The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $P(\cdot) = I(\cdot)$, where $I$ is the identity function):
$$\Phi(X, S, \mathcal{H}) = X^{(L)}$$
$$X^{(0)} = X \qquad X^{(l)} = P\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} S^kX^{(l - 1)}{H}_k^{(l)}\Bigg)\Bigg]$$

#### Random Graph Positional Encodings (R-PEARL)
The R-PEARL architecture extends on the GCN by instantiating it with simply one layer – a TAG Convolution and Demodulator. The mathematical equations below express the functionality of the R-PEARL network:
1. The white-noise matrix is sampled from the Gaussian distribution. $\renewcommand{\utilde}[1]{\underset{\sim}{#1}}$ $$Q \in \mathbb{R}^{M \times N} \qquad Q \sim \mathcal{N}(0, I) \qquad Q = \begin{bmatrix}
  \mathbf{q}^{(0)} & \cdots & \mathbf{q}^{(m)} & \cdots & \mathbf{q}^{(M)}
  \end{bmatrix}$$

2. The R-PEARL network has row-vector parameter $H^{(0)} \in \mathbb{R}^{1 \times D}$. It takes in each column of the white-noise matrix individually and produces a sample $P^{(m)} \in \mathbb{R}^{N \times D}$, which are then pooled to form GREP $P$:
$$P^{(m)} = \Phi\Big(\mathbf{q}^{(m)}, S, \mathcal{H}\Big) = \sigma\bigg(\sum_{k = 0}^{K = 1} S^k\mathbf{q}^{(m)} {H}_k\bigg)$$
$$P = \hat{\mathbb{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} P^{(m)}$$

#### Transformer

The Transformer architecture follows that of the Llama3.2-3B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $E$ and $\tilde{X}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } E = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } X = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$X = \tilde{X} + P$$

$${Z}_{1:t}^{(L)} = \operatorname{Trf}\bigg({X}_{1:t}, {\mathcal{T}}_l\bigg) \qquad {\mathcal{T}}_l = \begin{bmatrix}
{Q}_l & {K}_l & {V}_l & \left({W}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big({Z}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(E, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$


In [1]:
%env CUDA_VISIBLE_DEVICES=1

env: CUDA_VISIBLE_DEVICES=1


In [2]:
# Import modules.
import torch
import random

from prism.models import inference, gnn_llm
from prism.data import data, compact_prompt

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Setup the Gemma 4 model from Hugging Face.
llm = AutoModelForCausalLM.from_pretrained("google/gemma-4-12B-it", dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("google/gemma-4-12B-it")

# Initialize a barebones planner for testing.
graph_mask_llm = gnn_llm.GraphMaskLLM(llm, use_edges=False)
planner = inference.GraphAugmentedInMemoryLLM(graph_mask_llm, tokenizer, False)

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [4]:
from datasets import load_dataset

# Load in training dataset.
full_dataset = load_dataset("json", data_files=["../data/gen/nav100_n10_gemma_data/split/formatted_all_new_2turn__train.json"], split="train")
full_dataset = data.preprocess_dataset(
    full_dataset, tokenizer,
    architecture="graph_mask_llm",
    text_edge_list=False,
)

## Experiments

### §1 Testing a Raw `GraphMaskLLM`

In [5]:
# Query a random prompt.
msg = compact_prompt.spine_to_compact_messages(
    random.choice(full_dataset['conversations']),
    include_edges=False
)[:2]
msg

[{'role': 'system',
  'content': 'You are a navigation planner for a mobile robot. Below is a scene graph listing the environment\'s regions, its objects, and the robot\'s starting location. The connections between these nodes — which regions border one another, and which region each object is in — are available to you in latent space; reason over reachability and paths from that latent access even though the connecting edges are not written out here.\n\nAnswer in two parts, in this exact order:\n1. One <think> … </think> block that holds ALL of your reasoning. Begin it with "Relevant graph:" followed by the specific nodes this task depends on — never leave this blank — then "Reasoning:" followed by your step-by-step path-finding. Every bit of thinking goes inside this block, never after it.\n2. Immediately after </think>, give the final plan only: the route the robot follows, its nodes in order joined by arrows (for example, start_region -> middle_region -> goal_region). Put nothing e

In [6]:
planner.query_llm(msg)

[spine-llm] client=GraphAugmentedInMemoryLLM, prompt_tokens=37
[spine-llm] graph_found=False, n_graphs=0, robot_location=None
[spine-llm] raw_output (first 500 chars): To provide the specific path and edges, I need you to **provide the map, graph, or list of locations** (nodes) and the connections (edges) between them.

However, based on your request, here is the logical structure of how the path will be constructed once you provide the data.

### How the routing will work:
1.  **Start Node:** Starting Area
2.  **Intermediate Node:** Maintenance Bay (Required waypoint)
3.  **End Node:** Engine Room

### Example Format
If you provide a map like this:
*   *Start


('{"primary_goal": "", "relevant_graph": "", "reasoning": "", "plan": "[answer(To provide the specific path and edges, I need you to **provide the map, graph, or list of locations** (nodes) and the connections (edges) between them.\\n\\nHowever, based on your request, here is the logical structure of how the path will be constructed once you provide the data.\\n\\n### How the routing will work:\\n1.  **Start Node:** Starting Area\\n2.  **Intermediate Node:** Maintenance Bay (Required waypoint)\\n3.  **End Node:** Engine Room\\n\\n### Example Format\\nIf you provide a map like this:\\n*   *Starting Area $\\\\rightarrow$ Hallway A*\\n*   *Hallway A $\\\\rightarrow$ Maintenance Bay*\\n*   *Maintenance Bay $\\\\rightarrow$ Corridor B*\\n*   *Corridor B $\\\\rightarrow$ Engine Room*\\n\\n**The output I will provide will look like this:**\\n\\n*   **Full Path:** Starting Area $\\\\rightarrow$ Hallway A $\\\\rightarrow$ Maintenance Bay $\\\\rightarrow$ Corridor B $\\\\rightarrow$ Engine Room\